# Sprint 4 Mejora C: Descargar tarballs 006-012 y expandir HDF5

Este notebook descarga los 7 tarballs restantes del NIH ChestX-ray14
y agrega las imágenes directamente al HDF5 existente.

**Estrategia sin errores de Drive:**
- Descarga cada tarball a `/content/` (rápido, sin I/O de Drive).
- Lee las imágenes del tarball en memoria y las escribe al HDF5.
- Al final, copia el HDF5 ampliado a Drive (1 archivo, 1 operación).
- No se guardan PNGs sueltos en Drive en ningún momento.

**Tiempo estimado:** ~3-4 horas (descarga + procesamiento).
**Espacio en Drive:** ~3-4 GB adicionales para el HDF5 ampliado.
**Runtime:** CPU (no necesita GPU).


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

H5_LOCAL = '/content/nih_images.h5'
H5_DRIVE = f'{NIH_DIR}/nih_images.h5'

import os, sys, shutil, time, tarfile, io
import h5py
import numpy as np
from PIL import Image

!pip install -q h5py tqdm
from tqdm import tqdm

# Verificar HDF5 actual
if os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000:
    with h5py.File(H5_LOCAL, 'r') as hf:
        n_current = len(hf['images'])
    print(f'✓ HDF5 en disco local: {n_current:,} imágenes')
else:
    print('Copiando HDF5 desde Drive...')
    t0 = time.time()
    shutil.copy2(H5_DRIVE, H5_LOCAL)
    with h5py.File(H5_LOCAL, 'r') as hf:
        n_current = len(hf['images'])
    print(f'✓ Copiado ({(time.time()-t0)/60:.1f} min) | {n_current:,} imágenes')

import shutil as _sh
free_gb = _sh.disk_usage('/content').free / (1024**3)
print(f'Espacio libre en /content/: {free_gb:.1f} GB')


## 2. Descargar y procesar tarballs 006-012

Cada tarball:
1. Se descarga a `/content/` (~3.7 GB).
2. Se leen sus imágenes directamente en memoria (sin extraer a disco).
3. Se agregan al HDF5 local.
4. Se borra el tarball descargado para liberar espacio.

Esto evita que `/content/` se llene con 70,000 archivos PNG sueltos.


In [ ]:
BASE_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC'
TARBALLS = [f'images_{i:03d}.tar.gz' for i in range(6, 13)]  # 006 a 012
print(f'Tarballs a descargar: {TARBALLS}')

# Ver qué imágenes ya están en el HDF5
with h5py.File(H5_LOCAL, 'r') as hf:
    existing_in_h5 = set(hf['images'].keys())
print(f'Imágenes ya en HDF5: {len(existing_in_h5):,}')


In [ ]:
def download_and_add_to_hdf5(tarball_name: str, base_url: str,
                              h5_path: str, existing: set) -> tuple[int, int]:
    """
    Descarga un tarball, extrae imágenes en memoria y las agrega al HDF5.
    Devuelve (imágenes agregadas, imágenes fallidas).
    """
    url      = f'{base_url}/{tarball_name}'
    tar_local = f'/content/{tarball_name}'

    # ── Descarga ──────────────────────────────────────────────────
    if not (os.path.exists(tar_local) and
            os.path.getsize(tar_local) > 3_500_000_000 * 0.95):
        print(f'\nDescargando {tarball_name}...')
        ret = os.system(
            f'wget -c --progress=bar:force:noscroll -O "{tar_local}" "{url}"'
        )
        if ret != 0 or not os.path.exists(tar_local):
            print(f'✗ Error descargando {tarball_name}')
            return 0, 0
    else:
        print(f'\n✓ {tarball_name} ya descargado.')

    size_gb = os.path.getsize(tar_local) / (1024**3)
    print(f'  Tamaño: {size_gb:.2f} GB')

    # ── Leer y agregar al HDF5 ────────────────────────────────────
    saved, failed = 0, 0
    t0 = time.time()

    with tarfile.open(tar_local, 'r:gz') as tar,          h5py.File(h5_path, 'a') as hf:

        grp = hf.require_group('images')
        members = [m for m in tar.getmembers()
                   if m.isfile() and m.name.endswith('.png')]
        new_members = [m for m in members
                       if os.path.basename(m.name) not in existing]

        print(f'  Archivos en tar: {len(members):,} | '
              f'Nuevos para HDF5: {len(new_members):,}')

        for m in tqdm(new_members, desc=tarball_name, unit='img'):
            img_name = os.path.basename(m.name)
            try:
                f = tar.extractfile(m)
                if f is None:
                    continue
                with Image.open(io.BytesIO(f.read())) as pil_img:
                    arr = np.array(
                        pil_img.convert('L').resize((256, 256), Image.LANCZOS),
                        dtype=np.uint8
                    )
                grp.create_dataset(img_name, data=arr,
                                   compression='gzip', compression_opts=4)
                existing.add(img_name)
                saved += 1
            except Exception as e:
                failed += 1

    elapsed = time.time() - t0
    print(f'  ✓ Agregadas: {saved:,} | Fallidas: {failed} | '
          f'Tiempo: {elapsed/60:.1f} min')

    # ── Borrar tarball para liberar espacio ───────────────────────
    os.remove(tar_local)
    print(f'  Tarball borrado de /content/')

    return saved, failed

# ── Procesar todos los tarballs ───────────────────────────────────
total_added   = 0
total_failed  = 0
t_global = time.time()

for tb in TARBALLS:
    added, failed = download_and_add_to_hdf5(
        tb, BASE_URL, H5_LOCAL, existing_in_h5
    )
    total_added  += added
    total_failed += failed

    # Verificar estado actual del HDF5
    with h5py.File(H5_LOCAL, 'r') as hf:
        n_now = len(hf['images'])
    print(f'  HDF5 actual: {n_now:,} imágenes')

elapsed_total = time.time() - t_global
print()
print('=' * 60)
print(f'PROCESO COMPLETADO en {elapsed_total/60:.0f} min')
print(f'  Total agregadas: {total_added:,}')
print(f'  Total fallidas:  {total_failed:,}')
print(f'  HDF5 final:      {n_now:,} imágenes')


## 3. Actualizar CSV con todas las imágenes disponibles

In [ ]:
import pandas as pd

# Leer CSV completo del NIH (todos los labels)
df_full = pd.read_csv(f'{NIH_DIR}/Data_Entry_2017.csv')
print(f'CSV completo NIH: {len(df_full):,} filas')

# Filtrar a imágenes que están en el HDF5
with h5py.File(H5_LOCAL, 'r') as hf:
    in_h5 = set(hf['images'].keys())

df_subset = df_full[df_full['Image Index'].isin(in_h5)].copy()
print(f'Imágenes en HDF5: {len(in_h5):,}')
print(f'Filas en CSV que matchean: {len(df_subset):,}')

# Guardar CSV actualizado
subset_csv = f'{NIH_DIR}/Data_Entry_subset_local.csv'
df_subset.to_csv(subset_csv, index=False)
print(f'✓ CSV actualizado: {subset_csv}')

# Distribución de las 4 clases
print('\nDistribución (12 tarballs completos):')
print('=' * 55)
CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}
for cls in ['No Finding', 'Cardiomegaly', 'Effusion', 'Infiltration']:
    if cls == 'No Finding':
        n = (df_subset['Finding Labels'] == 'No Finding').sum()
    else:
        n = df_subset['Finding Labels'].str.contains(cls).sum()
    print(f'{cls:<20}: {n:>8,}')

# Pacientes únicos
def assign_label(finding):
    for kw, lbl in [('Infiltration',3),('Effusion',2),('Cardiomegaly',1)]:
        if kw in finding: return lbl
    if finding == 'No Finding': return 0
    return -1

df_subset['label'] = df_subset['Finding Labels'].apply(assign_label)
df_valid = df_subset[df_subset['label'] >= 0]
pat_labels = df_valid.groupby('Patient ID')['label'].max()

print('\nPacientes únicos por clase (define techo del balance):')
for lbl, name in CLASS_NAMES.items():
    n = (pat_labels == lbl).sum()
    print(f'  {name:<15}: {n:>6,} pacientes únicos')

ceiling = min((pat_labels == l).sum() for l in [1,2,3])
nf = min((pat_labels == 0).sum(), ceiling * 2)
print(f'\nTecho: {ceiling:,} | Dataset estimado:')
print(f'  {ceiling*3:,} positivas + {nf:,} No Finding = {ceiling*3+nf:,} pacientes')
print(f'  Train (todas imgs/paciente): ~{(ceiling*3+nf)*4:,} imágenes (estimado)')


## 4. Copiar HDF5 ampliado a Drive

In [ ]:
size_gb = os.path.getsize(H5_LOCAL) / (1024**3)
print(f'HDF5 local: {size_gb:.2f} GB')
print(f'Copiando a Drive...')

t0 = time.time()
shutil.copy2(H5_LOCAL, H5_DRIVE)
elapsed = time.time() - t0

print(f'✓ HDF5 copiado a Drive en {elapsed/60:.1f} min')
print(f'  Ruta: {H5_DRIVE}')
print(f'  Tamaño: {os.path.getsize(H5_DRIVE)/(1024**3):.2f} GB')
print()
print('Siguiente paso: S4_04_retrain_full.ipynb (T4 GPU)')
